# Lesson 7 — Communicating Uncertainty

Read `README.md` (or `README.pl.md`) first — it has Meridian Outlet's framing for today. You've chosen a threshold; now turn a raw probability into something a human can act on.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from task import load_and_merge_orders

df = load_and_merge_orders()
df.shape

## Same split, same model as Lessons 5-6

Reproduced exactly, so the probabilities you get here match what you've already seen.

In [ ]:
from task import fit_classifier, split_orders

train_df, test_df = split_orders(df)
model = fit_classifier(train_df)

## From a number to a category

Nobody in ops wants to read "0.34". Turn a few example probabilities into risk tiers first.

In [ ]:
from task import risk_tier

[risk_tier(p) for p in [0.05, 0.2, 0.4]]

## Build the report Meridian Outlet would actually see

One row per order: its id, its predicted probability, and its risk tier.

In [ ]:
from task import risk_report

report = risk_report(model, test_df)
report.head()

## Do the tiers actually reflect real risk?

Join the report back to what actually happened, and check the real return rate within each tier.

In [ ]:
combined = report.assign(actual=test_df["is_returned"].values)
combined.groupby("risk_tier")["actual"].agg(["count", "mean"]).reindex(["Low", "Medium", "High"])

## Are the tier probabilities themselves trustworthy?

The check above only looks at whether tiers are ordered correctly. It
doesn't ask whether the *number* attached to each tier means what it
claims — a "38% risk" prediction should correspond to roughly 38% of
those orders actually returning, not just "riskier than the other
tiers." That's calibration, and it's a different property from ranking.

In [ ]:
from task import tier_summary

tier_summary(model, test_df, "is_returned")

## One number for the whole model: Brier score

`tier_summary` only tells you about three coarse buckets. Brier score
summarizes calibration and discrimination together across every
prediction, with no binning at all — useful when you want a single
number to report, not a table.

In [ ]:
from sklearn.metrics import brier_score_loss
from task import brier_score

model_brier = brier_score(model, test_df, "is_returned")
baseline_prediction = train_df["is_returned"].mean()
baseline_brier = brier_score_loss(test_df["is_returned"], [baseline_prediction] * len(test_df))
model_brier, baseline_brier

## Your notes

Describe what you found when you checked the actual return rate by tier, and what `tier_summary`/`brier_score` told you about calibration — did the High tier's predicted probability match its observed rate? If not, would you trust an individual High-tier prediction at face value?